<a href="https://colab.research.google.com/github/Shea-Fyffe/transforming-personality-item-generation/blob/main/evaluate-DPO-dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluating DPO Preference Data

In [ ]:
#@title Install Libraries
!pip install "pynvml" "distilabel[hf-transformers, vllm]" "transformers" "torch" "huggingface_hub"

In [ ]:
#@title Load Libraries
from datasets import (Dataset, load_dataset)
from dataclasses import (dataclass, field, fields)
from huggingface_hub import login
import os
import re
import random
from typing import (Any, Callable, Dict, List, Literal, Optional, Union, TYPE_CHECKING)
from typing_extensions import override


from jinja2 import Template
from distilabel.models import (InferenceEndpointsLLM, TransformersLLM, vLLM)
from distilabel.pipeline import Pipeline
from distilabel.steps import (GlobalStep, StepInput)
from distilabel.steps.tasks.base import Task
from distilabel.typing import ChatType

from google.colab import (drive, userdata)



drive.mount('/content/drive')
login(token=userdata.get("HF_TOKEN"))
# !gcloud auth application-default login

Mounted at /content/drive


In [ ]:
@dataclass
class DPODataLoader:
    # init arguments
    data: Dataset
    prompt_column: str = "instruction"
    chosen_column: str = "chosen"
    rejected_column: str = "rejected"
    strip_text: bool = True
    strip_pattern: str = r'\n\n+'
    strip_replacement: str = '\n---\n'
    pad_outputs: bool = True
    pad_pre: str = '<|im_start|>'
    pad_suff: str = '<|im_end|>'

    @classmethod
    def from_json(cls, paths: str | list[str], **kwargs):
        _paths = cls._check_paths(paths, "json")
        _data = load_dataset("json", data_files=_paths)
        return cls(_data, **kwargs)

    @classmethod
    def from_csv(cls, paths: str | list[str], **kwargs):
        _paths = cls._check_paths(paths, "csv")
        _data = load_dataset("csv", data_files=_paths)
        return cls(_data, **kwargs)

    @classmethod
    def from_dict(cls, data: list[dict[str, any]], **kwargs):
        _data = Dataset.from_list(data)
        return cls(_data, **kwargs)

    @staticmethod
    def multmap(dataset: Dataset, columns: list[str], map_fun: Callable, fun_kwargs: dict[str, any]):
        """
        Modifies multiple columns in a dataset using Dataset.map()
        """
        def _map(examples):
            for col in columns:
                if col in examples:
                    examples[col] = [map_fun(x, **fun_kwargs) for x in examples[col]]
            return examples
        return dataset.map(_map, batched=True)

    @staticmethod
    def _check_paths(paths: str | list[str], file_type: str) -> list[str]:
        _paths = paths if isinstance(paths, list) else [paths]
        for _path in _paths:
            if isinstance(_path, str) & os.path.exists(_path):
                if not _path.endswith(file_type):
                    raise FileNotFoundError(f"Path {_path} is not a {file_type} file.")
                yield _path
            else:
                raise FileNotFoundError(f"Path {_path} not found.")

    @staticmethod
    def strip(text, strip_pattern, strip_replacement) -> str:
        """Strip pattern from text"""
        return re.sub(strip_pattern, strip_replacement, text)

    @staticmethod
    def pad(text, pad_pre: str, pad_suff: str) -> str:
        return f"{pad_pre}{text}{pad_suff}"

    def __post_init__(self):
        if isinstance(self.data, str):
            raise NotImplementedError("data argument invalid. Try using DPODataLoader.from_json or DPODataLoader.from_csv")
        if self.strip_text:
            self.data = self.multmap(
                dataset=self.data,
                columns=[self.prompt_column, self.chosen_column, self.rejected_column],
                map_fun=self.strip,
                fun_kwargs={"strip_pattern": self.strip_pattern, "strip_replacement": self.strip_replacement, }
                )
        if self.pad_outputs:
            self.data = self.multmap(
                dataset=self.data,
                columns=[self.chosen_column, self.rejected_column],
                map_fun=self.pad,
                fun_kwargs={"pad_pre": self.pad_pre, "pad_suff": self.pad_suff, }
                )

In [ ]:
class DPOEvaluationTask(Task):
    """Custom Evaluation Task

    Description:
        A lite version of distilabel.UltraFeedback
    """
    aspect: str = "overall-quality"
    system_prompt: str = (
            "Your role is to evaluate text quality based on given criteria.\n"
            'You\'ll receive an instructional description ("Instruction") and {no_texts} text outputs ("Text").\n'
            "Understand and interpret instructions to evaluate effectively.\n"
            "Provide a rating and rating rationale for for each text.\n"
            "The {no_texts} texts given are independent, and should be evaluated separately.\n"
        )
    template: Optional["Template"] = None

    def load(self) -> None:
        """Loads the Jinja2 template"""
        super().load()
        self._template = self.overall_rating_template

    def format_input(self, input: Dict[str, Any]) -> ChatType:
        """The input is formatted as a `ChatType` assuming that the instruction
        is the first interaction from the user within a conversation."""
        return [
            {
                "role": "system",
                "content": self.system_prompt.format(
                    no_texts=len(input["generations"])
                ),
            },
            {
                "role": "user",
                "content": self._template.render(  # type: ignore
                    instruction=input["instruction"], generations=input["generations"]
                ),
            },
        ]

    @override
    def format_output(  # type: ignore
        self, output: Union[str, None], input: Dict[str, Any] = None
    ) -> Dict[str, Any]:  # type: ignore
        """Pass raw output"""
        return {"output": output.strip()}

    @property
    def inputs(self) -> List[str]:
        """The input for the task is the `instruction`, and the `generations` for it."""
        return ["instruction", "generations"]

    @property
    def outputs(self) -> List[str]:
        return ["ratings", "rationales", "model_name"]

    @property
    def overall_rating_template(self) -> Template:
        template_overall_rating = """\
        # Text Quality Assessment

        Rate model outputs (1-5 scale) for uniqueness, formatting, and instruction following.

        Ratings for text outputs should be based on:
        - **Instruction Following**: Does the model's output align with given instructions and the user's intent?
        - **Output Formatting**: Is the output aligned with any output specifications described in the prompt?
        - **Uniqueness**: Does the output provide information that is not redundant or repetative?

        **Scoring**: Rating scores should range from 1 to 5, where:
        1. **Low Quality**: Contains redundancy, poor formatting, and ignores instructions.
        2. **Moderate Quality**: Contains redundancy, okay formatting, and follows minimal instructions.
        3. **Good**: Contains some redundancy, well formatted, and follow partial instructions.
        4. **Very Good**: Contains little redundancy, well formatted, and follows instructions.
        5. **Excellent**: Contains no redundancy, perfectly formatted, and follows all instructions.

        ## Format:

        ### Input
        Instruction: {{ instruction }}

        Texts:
        {%- for generation in generations %}
        <text {{ loop.index }}> {{ generation }}
        {%- endfor %}

        ### Output
        {%- for generation in generations %}
        #### Output for Text {{ loop.index }}
        Rating: [Rating]
        Rationale: [Concise rationale for the rating.]
        {%- endfor %}


        ## Constraints:
        - Minimize commentary about the rating process.
        - Output only Ratings and Rationale described in the Output section."""
        return Template(template_overall_rating)

class ShuffleStep(GlobalStep):
    @property
    def inputs(self) -> list[str]:
        return ["instruction", "chosen", "rejected"]
    @property
    def outputs(self) -> list[str]:
        return ["instruction", "generations", "order"]
    def process(self, inputs: StepInput) -> "StepOutput":
        outputs = []
        for input in inputs:
            chosen = input["chosen"]
            rejected = input["rejected"]
            pair = [chosen, rejected]
            random.shuffle(pair)
            order = ["chosen" if x == chosen else "rejected" for x in pair]
            outputs.append({"instruction": input["instruction"], "generations": pair, "order": order})
        yield outputs

In [ ]:
@dataclass
class DPOEvaluator:
    # init arguments
    data: DPODataLoader
    batch_size: int = 16
    feedback_model: str = "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B"
    feedback_model_kwargs: dict[str, any] = field(default_factory=dict)

    def _setup_data(self):
        if "prompt" in self.data["train"].column_names:
            self.data = self.data.rename_column("prompt", "instruction")
        self._data = self.data["train"]

    def _setup_pipeline(self):
        with Pipeline() as _pipeline:
            shuffler = ShuffleStep()
            evaluator = self.feedback_task_setup

            shuffler.connect(evaluator)

        self.pipeline = _pipeline
        return None

    ## Post-Init
    def __post_init__(self):
        self._setup_data()
        self._setup_pipeline()

    def evaluate(self, **kwargs):
        self.result = []
        self.result = self.pipeline.run(dataset=self._data,
                                        dataset_batch_size = self.batch_size, **kwargs)
        return self.result

    @property
    def setup_feedback_model(self):
        return vLLM(model = self.feedback_model, **self.feedback_model_defaults)

    @property
    def feedback_task_setup(self) -> DPOEvaluationTask:
        return DPOEvaluationTask(**self.feedback_task_defaults)

    @property
    def feedback_model_defaults(self) -> dict[str, any]:
        return {"quantization": "fp8",
                "generation_kwargs": {"max_new_tokens": 768},
                "extra_kwargs": {"tensor_parallel_size": 1, "max_model_len": 4096, "enforce_eager": True},} | self.feedback_model_kwargs

    @property
    def feedback_task_defaults(self) -> dict[str, any]:
        return {"llm":self.setup_feedback_model, "input_batch_size": self.batch_size}

In [ ]:
#@title Load Data
JSON_PATH = "/content/drive/My Drive/Academic/research/dissertation/NLP-in-Personality/data/study1/dpo-training-main-data.json"
DPOdata = DPODataLoader.from_json(JSON_PATH)

In [ ]:
#@title Instantiate Evaluator
DPOobj = DPOEvaluator(DPOdata.data)

In [ ]:
#@title Run Evaluator
RESULT = DPOobj.evaluate()

In [ ]:
#@title Output Data
RESULT["default"]['train'].to_csv("dpo-eval-results.csv")